# Agent 2: agent z pamięcią i narzędziami

Ten notebook pokazuje **średni poziom złożoności**:
- agent ma krótką pamięć rozmowy,
- wybiera narzędzie,
- wykonuje prosty workflow,
- dopiero potem formułuje odpowiedź przez model.

## Narzędzia
- kalkulator
- wyszukiwarka lokalnych notatek (mini-RAG)
- generator listy zadań

Ten wzorzec dobrze pokazuje studentom przejście:
**LLM -> tool selection -> execution -> answer synthesis**

In [7]:
!apt-get update -y
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,143 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,965 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,226 kB]
G

In [8]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [9]:
import time
time.sleep(8)

In [10]:
!ollama pull gemma2:2b

In [11]:
USE_OLLAMA = True   # zmień na True, jeśli masz uruchomione Ollama lokalnie
OLLAMA_MODEL = "gemma2:2b"

In [12]:
from dataclasses import dataclass, field
from typing import List, Dict, Callable, Any
import math
import re
from collections import Counter

In [13]:
class MockLLM:
    def generate(self, prompt: str) -> str:
        if "narzędzie" in prompt.lower():
            if any(x in prompt.lower() for x in ["2+2", "oblicz", "policz", "suma"]):
                return "calculator"
            if any(x in prompt.lower() for x in ["notat", "rag", "dokument", "źródł"]):
                return "notes_search"
            return "task_list"
        return "Na podstawie działania narzędzia agent przygotował końcową odpowiedź."

class OllamaLLM:
    def __init__(self, model: str, url: str = "http://localhost:11434/api/generate"):
        self.model = model
        self.url = url
    def generate(self, prompt: str) -> str:
        import requests
        payload = {"model": self.model, "prompt": prompt, "stream": False}
        resp = requests.post(self.url, json=payload, timeout=120)
        resp.raise_for_status()
        return resp.json()["response"]

def get_llm(use_ollama: bool, model: str):
    return OllamaLLM(model) if use_ollama else MockLLM()

In [14]:
# Mini baza wiedzy do lokalnego wyszukiwania
NOTES = [
    "RAG to Retrieval-Augmented Generation: najpierw wyszukiwanie, potem generacja.",
    "Chunking dzieli dokumenty na fragmenty, które można indeksować i przeszukiwać.",
    "Embedding reprezentuje tekst jako wektor liczb do porównywania podobieństwa.",
    "Agent to system, który może planować, wybierać narzędzia i wykonywać akcje."
]

def tokenize(text: str):
    return re.findall(r"\w+", text.lower())

def notes_search(query: str, top_k: int = 2):
    q = Counter(tokenize(query))
    scored = []
    for note in NOTES:
        n = Counter(tokenize(note))
        overlap = sum((q & n).values())
        scored.append((overlap, note))
    scored.sort(reverse=True, key=lambda x: x[0])
    return [note for score, note in scored[:top_k] if score > 0]

def calculator(expression: str):
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        raise ValueError("Niedozwolone znaki w wyrażeniu.")
    return eval(expression, {"__builtins__": {}}, {})

def task_list(goal: str):
    return [
        f"Zrozumieć cel: {goal}",
        "Rozbić problem na kroki",
        "Wykonać najważniejszy krok",
        "Podsumować wynik"
    ]

In [15]:
@dataclass
class ToolAgent:
    llm: Any
    memory: List[Dict[str, str]] = field(default_factory=list)

    def choose_tool(self, user_input: str) -> str:
        prompt = f"""Wybierz jedno narzędzie dla zadania użytkownika.
Dostępne narzędzia: calculator, notes_search, task_list.
Zwróć tylko nazwę narzędzia.

Zadanie: {user_input}"""
        return self.llm.generate(prompt).strip()

    def use_tool(self, tool_name: str, user_input: str):
        if tool_name == "calculator":
            expr_match = re.search(r"[0-9\+\-\*/\(\)\. ]+", user_input)
            expr = expr_match.group(0).strip() if expr_match else "0"
            return {"tool": tool_name, "result": calculator(expr)}
        if tool_name == "notes_search":
            return {"tool": tool_name, "result": notes_search(user_input)}
        return {"tool": "task_list", "result": task_list(user_input)}

    def run(self, user_input: str) -> Dict[str, Any]:
        self.memory.append({"role": "user", "content": user_input})
        tool = self.choose_tool(user_input)
        tool_output = self.use_tool(tool, user_input)

        summary_prompt = f"""Użytkownik: {user_input}
Wybrane narzędzie: {tool_output['tool']}
Wynik narzędzia: {tool_output['result']}

Napisz krótką odpowiedź dla użytkownika po polsku."""
        answer = self.llm.generate(summary_prompt)
        self.memory.append({"role": "assistant", "content": answer})
        return {"tool": tool, "tool_output": tool_output, "answer": answer}

In [16]:
agent = ToolAgent(llm=get_llm(USE_OLLAMA, OLLAMA_MODEL))

case1 = agent.run("Oblicz 2+2*5")
case2 = agent.run("Znajdź notatki o RAG i embeddingach")
case3 = agent.run("Przygotuj plan pracy nad mini projektem NLP")

print(case1)
print("-" * 80)
print(case2)
print("-" * 80)
print(case3)

assert case1["tool"] == "calculator"
assert case2["tool"] in {"notes_search", "task_list"}  # mock może wybrać jedną z prostych ścieżek
assert isinstance(case3["answer"], str)
assert len(agent.memory) >= 6

{'tool': 'calculator', 'tool_output': {'tool': 'calculator', 'result': 12}, 'answer': 'Twoje obliczenia dały wynik 12! \n'}
--------------------------------------------------------------------------------
{'tool': 'notes_search', 'tool_output': {'tool': 'notes_search', 'result': ['RAG to Retrieval-Augmented Generation: najpierw wyszukiwanie, potem generacja.', 'Chunking dzieli dokumenty na fragmenty, które można indeksować i przeszukiwać.']}, 'answer': 'W skrócie, RAG (Retrieval-Augmented Generation) to technika wykorzystująca wyszukiwanie do znalezienia informacji w dokumencie. Po znalezieniu informacji, RAG wykorzystuje je do generowania odpowiedzi. \n\nChunking jest innym przykładem, w którym dokumenty są dzielone na fragmenty, które można indeksować i przeszukiwać, co wpływa na szybkość generowania odpowiedzi.\n\n\n'}
--------------------------------------------------------------------------------
{'tool': 'task_list', 'tool_output': {'tool': 'task_list', 'result': ['Zrozumieć cel:

## Co pokazuje ten agent
- pamięć krótkoterminową,
- routing do narzędzi,
- prostą architekturę agentową bez płatnego API,
- gotowy punkt wyjścia do LangChain / LlamaIndex / własnego frameworka.